In [1]:
from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd
import librosa
import torch
from tqdm.auto import tqdm
from transformers import AutoFeatureExtractor, Wav2Vec2Model
from joblib import Parallel, delayed

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
BASE = Path("/content/drive/MyDrive/pruebas/pre/embeddings")
WIN_DIR = BASE / "windows_vfinal"
OUT_DIR = BASE / "embeddings_w2v"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_WIN = WIN_DIR / "windows_train.npz"
VAL_WIN   = WIN_DIR / "windows_val.npz"
TEST_WIN  = WIN_DIR / "windows_test.npz"

SR = 16000
MODEL_ID = "facebook/wav2vec2-large-xlsr-53"
POOLING = "meanstd"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE: cuda
GPU: Tesla T4


In [4]:
torch.set_grad_enabled(False)

fe = AutoFeatureExtractor.from_pretrained(MODEL_ID)
w2v = Wav2Vec2Model.from_pretrained(MODEL_ID).to(DEVICE).eval()

FRAME_STRIDE = int(np.prod(getattr(w2v.config, "conv_stride", [320])))
print("FRAME_STRIDE:", FRAME_STRIDE)
print("Hidden size:", w2v.config.hidden_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FRAME_STRIDE: 320
Hidden size: 1024


In [5]:
def load_windows_npz(npz_path: Path) -> pd.DataFrame:
    D = np.load(npz_path, allow_pickle=True)

    df = pd.DataFrame({
        "path": D["path"].astype("U"),
        "orig_path": D["orig_path"].astype("U") if "orig_path" in D else D["path"].astype("U"),
        "clip_id": D["clip_id"].astype("U") if "clip_id" in D else np.array([""] * len(D["path"])),
        "speaker_id_final": D["speaker_id_final"].astype("U") if "speaker_id_final" in D else np.array([""] * len(D["path"])),
        "clip_idx": D["clip_idx"].astype(np.int64),
        "start_samp": D["start_samp"].astype(np.int64),
        "end_samp": D["end_samp"].astype(np.int64),
        "valence": D["valence"].astype(np.float32),
        "arousal": D["arousal"].astype(np.float32),
        "dominance": D["dominance"].astype(np.float32),
        "win_rms": D["win_rms"].astype(np.float32),
        "zrms_log": D["zrms_log"].astype(np.float32),
        "lang": D["lang"].astype("U"),
        "is_aug": D["is_aug"].astype(bool) if "is_aug" in D else np.zeros(len(D["path"]), dtype=bool),
        "aug_ops": D["aug_ops"].astype("U") if "aug_ops" in D else np.array([""] * len(D["path"])),
    })

    df = df.sort_values(["path", "start_samp"]).reset_index(drop=True)
    return df

In [6]:
def safe_load_audio(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    y = y.astype(np.float32)
    y = y - y.mean()
    return y

In [ ]:
@torch.no_grad()
def forward_hidden(y: np.ndarray) -> np.ndarray:
    inputs = fe([y], sampling_rate=SR, return_tensors="pt", padding=True)
    iv = inputs["input_values"].to(DEVICE)
    am = inputs.get("attention_mask")
    if am is not None:
        am = am.to(DEVICE)

    H = w2v(input_values=iv, attention_mask=am).last_hidden_state.squeeze(0)
    H_np = H.detach().cpu().numpy().astype(np.float32)

    del inputs, iv, am, H
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return H_np

In [ ]:
def pool_windows_meanstd_from_frames(H_np: np.ndarray, starts: np.ndarray, ends: np.ndarray) -> np.ndarray:
    Tprime, D = H_np.shape

    s_f = np.floor(starts / FRAME_STRIDE).astype(np.int64)
    e_f = np.ceil(ends / FRAME_STRIDE).astype(np.int64)

    s_f = np.clip(s_f, 0, max(Tprime - 1, 0))
    e_f = np.clip(e_f, 1, Tprime)

    cs = np.cumsum(H_np, axis=0)
    cs2 = np.cumsum(H_np * H_np, axis=0)

    if POOLING == "mean":
        X = np.empty((len(s_f), D), dtype=np.float32)
    else:
        X = np.empty((len(s_f), 2 * D), dtype=np.float32)

    for i, (s, e) in enumerate(zip(s_f, e_f)):
        if e <= s:
            e = min(s + 1, Tprime)

        L = float(e - s)
        sum_ = cs[e - 1] - (cs[s - 1] if s > 0 else 0.0)
        mean = sum_ / L

        if POOLING == "mean":
            X[i, :] = mean.astype(np.float32)
        else:
            sum2 = cs2[e - 1] - (cs2[s - 1] if s > 0 else 0.0)
            var = np.maximum(sum2 / L - mean * mean, 1e-12)
            std = np.sqrt(var, dtype=np.float32)
            X[i, :D] = mean.astype(np.float32)
            X[i, D:] = std

    return X

In [ ]:
def extract_embeddings_from_windows(npz_in: Path, split_name="train"):
    print(f"\n Extrayendo embeddings: {split_name} ==")
    dfw = load_windows_npz(npz_in)

    X_list = []
    meta_parts = []

    grouped = dfw.groupby("path", sort=False)

    for wav, g in tqdm(grouped, total=dfw["path"].nunique(), desc=f"W2V {split_name}"):
        y = safe_load_audio(wav, sr=SR)
        H = forward_hidden(y)

        starts = g["start_samp"].to_numpy(np.int64)
        ends = g["end_samp"].to_numpy(np.int64)

        Xw = pool_windows_meanstd_from_frames(H, starts, ends)

        X_list.append(Xw)
        meta_parts.append(g.copy())

        del y, H, Xw
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    meta = pd.concat(meta_parts, axis=0).reset_index(drop=True)

    assert len(meta) == len(X), "Meta y embeddings quedaron desalineados"

    return X, meta

In [7]:
def save_embeddings_npz(X: np.ndarray, meta: pd.DataFrame, out_path: Path):
    np.savez_compressed(
        out_path,
        Xw=X,
        path=np.array(meta["path"].astype(str).tolist()),
        orig_path=np.array(meta["orig_path"].astype(str).tolist()),
        clip_id=np.array(meta["clip_id"].astype(str).tolist()),
        speaker_id_final=np.array(meta["speaker_id_final"].astype(str).tolist()),
        clip_idx=meta["clip_idx"].to_numpy(np.int64),
        start_samp=meta["start_samp"].to_numpy(np.int64),
        end_samp=meta["end_samp"].to_numpy(np.int64),
        valence=meta["valence"].to_numpy(np.float32),
        arousal=meta["arousal"].to_numpy(np.float32),
        dominance=meta["dominance"].to_numpy(np.float32),
        win_rms=meta["win_rms"].to_numpy(np.float32),
        zrms_log=meta["zrms_log"].to_numpy(np.float32),
        lang=np.array(meta["lang"].astype(str).tolist()),
        is_aug=meta["is_aug"].to_numpy(bool),
        aug_ops=np.array(meta["aug_ops"].astype(str).tolist()),
        sr=np.int64(SR),
        pooling=np.bytes_(POOLING.encode("utf-8")),
        frame_stride=np.int64(FRAME_STRIDE),
    )
    print("Guardado:", out_path, "| X shape:", X.shape)

In [ ]:
X_va, meta_va = extract_embeddings_from_windows(VAL_WIN, split_name="val")

In [ ]:
save_embeddings_npz(X_va, meta_va, OUT_DIR / "val_embeddings_w2v.npz")

Guardado: /content/drive/MyDrive/pruebas/pre/embeddings/embeddings_w2v/val_embeddings_w2v.npz | X shape: (8459, 2048)


In [ ]:
X_va.shape

(8459, 2048)

In [ ]:
meta_va.head()

,path,orig_path,clip_id,speaker_id_final,clip_idx,start_samp,end_samp,valence,arousal,dominance,win_rms,zrms_log,lang,is_aug,aug_ops
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio1_1-02-03-03,Audio1,5,0,32000,2.0,3.0,3.0,0.275379,1.868954,es,False,
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio1_1-02-03-03,Audio1,5,16000,48000,2.0,3.0,3.0,0.279429,1.908437,es,False,
2,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio1_1-02-03-03,Audio1,5,32000,64000,2.0,3.0,3.0,0.238039,1.498897,es,False,
3,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio1_1-02-03-03,Audio1,5,48000,80000,2.0,3.0,3.0,0.266957,1.786448,es,False,
4,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio1_10-02-03-04,Audio1,0,0,32000,2.0,3.0,4.0,0.192927,1.036640,es,False,


In [ ]:
@torch.no_grad()
def forward_hidden_batch(wavs: list[np.ndarray]) -> list[np.ndarray]:
    inputs = fe(
        wavs,
        sampling_rate=SR,
        return_tensors="pt",
        padding=True
    )

    iv = inputs["input_values"].to(DEVICE)
    am = inputs.get("attention_mask")
    if am is not None:
        am = am.to(DEVICE)

    H = w2v(input_values=iv, attention_mask=am).last_hidden_state  # [B, T', D]
    H_np = H.detach().cpu().numpy().astype(np.float32)

    # longitudes válidas aproximadas en frames del encoder
    hidden_list = []
    if am is not None:
        valid_samples = am.sum(dim=1).detach().cpu().numpy()
        valid_frames = np.ceil(valid_samples / FRAME_STRIDE).astype(int)

        for i in range(H_np.shape[0]):
            t = max(1, min(H_np.shape[1], valid_frames[i]))
            hidden_list.append(H_np[i, :t])
    else:
        for i in range(H_np.shape[0]):
            hidden_list.append(H_np[i])

    del inputs, iv, am, H
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return hidden_list

In [ ]:
def extract_embeddings_from_windows_sharded_fast(
    npz_in: Path,
    out_prefix: Path,
    shard_nwins: int = 12000,
    batch_audios: int = 8,
    resume: bool = True
):
    shards_dir = out_prefix.parent / f"{out_prefix.stem}_shards"
    shards_dir.mkdir(parents=True, exist_ok=True)

    index_json = shards_dir / "index.json"
    done_json = shards_dir / "done_paths.json"

    print(f"\n== Extrayendo embeddings desde {npz_in.name} ==")
    dfw = load_windows_npz(npz_in)

    done_paths = set()
    shards_idx = {"meta": {}, "shards": []}

    if resume:
        if done_json.exists():
            try:
                done_paths = set(json.loads(done_json.read_text()))
            except Exception:
                done_paths = set()

        if index_json.exists():
            try:
                shards_idx = json.loads(index_json.read_text())
            except Exception:
                shards_idx = {"meta": {}, "shards": []}

    grouped_items = [(wav, g) for wav, g in dfw.groupby("path", sort=False) if wav not in done_paths]

    X_buf = []
    meta_buf = []
    n_buf = 0

    total_written = sum(s.get("nrows", 0) for s in shards_idx.get("shards", []))
    shard_id = len(shards_idx.get("shards", []))

    def _flush():
        nonlocal X_buf, meta_buf, n_buf, shard_id, total_written, shards_idx

        if not X_buf:
            return

        Xcat = np.concatenate(X_buf, axis=0).astype(np.float32)
        meta_cat = pd.concat(meta_buf, axis=0).reset_index(drop=True)

        shard_path = shards_dir / f"shard_{shard_id:05d}.npz"

        np.savez_compressed(
            shard_path,
            Xw=Xcat,
            path=np.array(meta_cat["path"].astype(str).tolist()),
            orig_path=np.array(meta_cat["orig_path"].astype(str).tolist()),
            clip_id=np.array(meta_cat["clip_id"].astype(str).tolist()),
            speaker_id_final=np.array(meta_cat["speaker_id_final"].astype(str).tolist()),
            clip_idx=meta_cat["clip_idx"].to_numpy(np.int64),
            start_samp=meta_cat["start_samp"].to_numpy(np.int64),
            end_samp=meta_cat["end_samp"].to_numpy(np.int64),
            valence=meta_cat["valence"].to_numpy(np.float32),
            arousal=meta_cat["arousal"].to_numpy(np.float32),
            dominance=meta_cat["dominance"].to_numpy(np.float32),
            win_rms=meta_cat["win_rms"].to_numpy(np.float32),
            zrms_log=meta_cat["zrms_log"].to_numpy(np.float32),
            lang=np.array(meta_cat["lang"].astype(str).tolist()),
            is_aug=meta_cat["is_aug"].to_numpy(bool),
            aug_ops=np.array(meta_cat["aug_ops"].astype(str).tolist()),
        )

        shards_idx["shards"].append({
            "path": str(shard_path),
            "nrows": int(Xcat.shape[0]),
        })

        shard_id += 1
        total_written += int(Xcat.shape[0])

        X_buf = []
        meta_buf = []
        n_buf = 0

        shards_idx["meta"] = {
            "npz_source": str(npz_in),
            "sr": int(SR),
            "pooling": POOLING,
            "device": DEVICE,
            "frame_stride": int(FRAME_STRIDE),
            "total_written": int(total_written),
            "batch_audios": int(batch_audios),
        }

        index_json.write_text(json.dumps(shards_idx, indent=2))
        done_json.write_text(json.dumps(sorted(list(done_paths))))

        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    pbar = tqdm(range(0, len(grouped_items), batch_audios), desc="W2V batch-shards")

    for start_idx in pbar:
        batch = grouped_items[start_idx:start_idx + batch_audios]

        wavs = []
        metas = []

        for wav, g in batch:
            y = safe_load_audio(wav, sr=SR)
            wavs.append(y)
            metas.append((wav, g))

        H_list = forward_hidden_batch(wavs)

        for H_np, (wav, g) in zip(H_list, metas):
            starts = g["start_samp"].to_numpy(np.int64)
            ends = g["end_samp"].to_numpy(np.int64)

            Xw = pool_windows_meanstd_from_frames(H_np, starts, ends)

            X_buf.append(Xw)
            meta_buf.append(g)
            n_buf += Xw.shape[0]
            done_paths.add(wav)

        if n_buf >= shard_nwins:
            _flush()
            pbar.set_postfix_str(f"guardado={total_written}")

        del wavs, metas, H_list

    _flush()

    print(f"[OK] Shards en {shards_dir} | total ventanas={total_written}")
    return shards_dir

In [8]:
def merge_shards_to_npz(index_json: Path, out_npz: Path):
    idx = json.loads(Path(index_json).read_text())
    shards = idx.get("shards", [])
    if not shards:
        raise RuntimeError("No hay shards en el índice.")

    Xs = []
    metas = []

    for s in shards:
        D = np.load(s["path"], allow_pickle=True)

        Xs.append(D["Xw"])

        metas.append(pd.DataFrame({
            "path": D["path"].astype("U"),
            "orig_path": D["orig_path"].astype("U"),
            "clip_id": D["clip_id"].astype("U"),
            "speaker_id_final": D["speaker_id_final"].astype("U"),
            "clip_idx": D["clip_idx"].astype(np.int64),
            "start_samp": D["start_samp"].astype(np.int64),
            "end_samp": D["end_samp"].astype(np.int64),
            "valence": D["valence"].astype(np.float32),
            "arousal": D["arousal"].astype(np.float32),
            "dominance": D["dominance"].astype(np.float32),
            "win_rms": D["win_rms"].astype(np.float32),
            "zrms_log": D["zrms_log"].astype(np.float32),
            "lang": D["lang"].astype("U"),
            "is_aug": D["is_aug"].astype(bool),
            "aug_ops": D["aug_ops"].astype("U"),
        }))

    Xw = np.concatenate(Xs, axis=0).astype(np.float32)
    meta = pd.concat(metas, axis=0).reset_index(drop=True)

    np.savez_compressed(
        out_npz,
        Xw=Xw,
        path=np.array(meta["path"].astype(str).tolist()),
        orig_path=np.array(meta["orig_path"].astype(str).tolist()),
        clip_id=np.array(meta["clip_id"].astype(str).tolist()),
        speaker_id_final=np.array(meta["speaker_id_final"].astype(str).tolist()),
        clip_idx=meta["clip_idx"].to_numpy(np.int64),
        start_samp=meta["start_samp"].to_numpy(np.int64),
        end_samp=meta["end_samp"].to_numpy(np.int64),
        valence=meta["valence"].to_numpy(np.float32),
        arousal=meta["arousal"].to_numpy(np.float32),
        dominance=meta["dominance"].to_numpy(np.float32),
        win_rms=meta["win_rms"].to_numpy(np.float32),
        zrms_log=meta["zrms_log"].to_numpy(np.float32),
        lang=np.array(meta["lang"].astype(str).tolist()),
        is_aug=meta["is_aug"].to_numpy(bool),
        aug_ops=np.array(meta["aug_ops"].astype(str).tolist()),
        sr=np.int64(SR),
        pooling=np.bytes_(POOLING.encode("utf-8")),
        frame_stride=np.int64(FRAME_STRIDE),
    )

    print(f"[OK] Final guardado en: {out_npz} | X shape: {Xw.shape}")

In [ ]:
test_shards = extract_embeddings_from_windows_sharded_fast(
    TEST_WIN,
    OUT_DIR / "test_embeddings_w2v",
    shard_nwins=12000,
    batch_audios=4,
    resume=True
)


== Extrayendo embeddings desde windows_test.npz ==


W2V batch-shards:   0%|          | 0/792 [00:00<?, ?it/s]

[OK] Shards en /content/drive/MyDrive/pruebas (1)/pre/embeddings/embeddings_w2v/test_embeddings_w2v_shards | total ventanas=9980


In [ ]:
merge_shards_to_npz(
    test_shards / "index.json",
    OUT_DIR / "test_embeddings_w2v.npz"
)

[OK] Final guardado en: /content/drive/MyDrive/pruebas (1)/pre/embeddings/embeddings_w2v/test_embeddings_w2v.npz | X shape: (9980, 2048)


In [ ]:
OUT_DIR

PosixPath('/content/drive/MyDrive/pruebas (1)/pre/embeddings/embeddings_w2v')

In [ ]:
train_shards = extract_embeddings_from_windows_sharded_fast(
    TRAIN_WIN,
    OUT_DIR / "train_embeddings_w2v",
    shard_nwins=12000,
    batch_audios=4,
    resume=True
)


== Extrayendo embeddings desde windows_train.npz ==


W2V batch-shards:   0%|          | 0/2966 [00:00<?, ?it/s]

/tmp/ipykernel_1223/2422485145.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(path, sr=sr, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/pruebas/pre/train_aug/es/Audio10_1-02-02-04_aug1_rvb0.14s_0.09.wav'

In [ ]:
import os

path = "/content/drive/MyDrive/pruebas/pre/train_aug/es/Audio10_1-02-02-04_aug1_rvb0.14s_0.09.wav"

print("exists:", os.path.exists(path))

exists: True


In [ ]:
train_shards = extract_embeddings_from_windows_sharded_fast(
    TRAIN_WIN,
    OUT_DIR / "train_embeddings_w2v",
    shard_nwins=12000,
    batch_audios=4,
    resume=True
)


== Extrayendo embeddings desde windows_train.npz ==


W2V batch-shards:   0%|          | 0/1124 [00:00<?, ?it/s]

[OK] Shards en /content/drive/MyDrive/pruebas/pre/embeddings/embeddings_w2v/train_embeddings_w2v_shards | total ventanas=38249


In [ ]:
idx = json.loads((train_shards / "index.json").read_text())

In [ ]:
for s in idx["shards"]:
    s["path"] = s["path"].replace("pruebas (1)", "pruebas")

# corregido
(train_shards / "index.json").write_text(json.dumps(idx, indent=2))

900

In [ ]:
merge_shards_to_npz(
    train_shards / "index.json",
    OUT_DIR / "train_embeddings_w2v.npz"
)

[OK] Final guardado en: /content/drive/MyDrive/pruebas/pre/embeddings/embeddings_w2v/train_embeddings_w2v.npz | X shape: (38249, 2048)


In [ ]:
print(idx["shards"])

[{'path': '/content/drive/MyDrive/pruebas/pre/embeddings/embeddings_w2v/train_embeddings_w2v_shards/shard_00000.npz', 'nrows': 12007}, {'path': '/content/drive/MyDrive/pruebas/pre/embeddings/embeddings_w2v/train_embeddings_w2v_shards/shard_00001.npz', 'nrows': 12006}, {'path': '/content/drive/MyDrive/pruebas/pre/embeddings/embeddings_w2v/train_embeddings_w2v_shards/shard_00002.npz', 'nrows': 12005}, {'path': '/content/drive/MyDrive/pruebas/pre/embeddings/embeddings_w2v/train_embeddings_w2v_shards/shard_00003.npz', 'nrows': 2231}]


In [ ]:
D = np.load(OUT_DIR / "train_embeddings_w2v.npz", allow_pickle=True)
print(D.files)
print("Xw:", D["Xw"].shape)
print("zrms_log:", D["zrms_log"].shape)
print("valence:", D["valence"].shape)

['Xw', 'path', 'orig_path', 'clip_id', 'speaker_id_final', 'clip_idx', 'start_samp', 'end_samp', 'valence', 'arousal', 'dominance', 'win_rms', 'zrms_log', 'lang', 'is_aug', 'aug_ops', 'sr', 'pooling', 'frame_stride']
Xw: (38249, 2048)
zrms_log: (38249,)
valence: (38249,)


features adicionales

In [9]:
BASE = Path("/content/drive/MyDrive/pruebas/pre")
WIN_DIR = BASE / "embeddings" / "windows_vfinal"
OUT_DIR = BASE / "spectral_feats"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_WIN = WIN_DIR / "windows_train.npz"
VAL_WIN   = WIN_DIR / "windows_val.npz"
TEST_WIN  = WIN_DIR / "windows_test.npz"

SR = 16000

print(TRAIN_WIN)
print(VAL_WIN)
print(TEST_WIN)

/content/drive/MyDrive/pruebas/pre/embeddings/windows_vfinal/windows_train.npz
/content/drive/MyDrive/pruebas/pre/embeddings/windows_vfinal/windows_val.npz
/content/drive/MyDrive/pruebas/pre/embeddings/windows_vfinal/windows_test.npz


In [8]:
def compute_spectral_features_for_audio(y, sr, starts, ends, n_fft=400, hop_length=160):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))

    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=sr)[0]

    feats = np.zeros((len(starts), 4), dtype=np.float32)

    for i, (s, e) in enumerate(zip(starts, ends)):
        f0 = int(s // hop_length)
        f1 = int(np.ceil(e / hop_length))
        f1 = max(f1, f0 + 1)

        c = centroid[f0:f1]
        b = bandwidth[f0:f1]

        if len(c) > 0 and len(b) > 0:
            feats[i, 0] = np.mean(c)
            feats[i, 1] = np.std(c)
            feats[i, 2] = np.mean(b)
            feats[i, 3] = np.std(b)

    return feats

In [10]:
def process_one_audio_spectral(wav, g, sr=SR):
    try:
        y = safe_load_audio(wav, sr=sr)

        starts = g["start_samp"].to_numpy(np.int64)
        ends = g["end_samp"].to_numpy(np.int64)

        Xspec = compute_spectral_features_for_audio(y, sr, starts, ends)

        return Xspec, g.copy()

    except Exception as e:
        print(f"[WARN] spectral failed: {wav} | {e}")
        return None, None

In [11]:
def extract_spectral_features_fast(npz_in: Path, split_name="train", n_jobs=2):
    print(f"\n== Spectral features desde {npz_in.name} ==")

    dfw = load_windows_npz(npz_in)
    grouped_items = list(dfw.groupby("path", sort=False))

    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(process_one_audio_spectral)(wav, g)
        for wav, g in tqdm(grouped_items, desc=f"Spectral {split_name}")
    )

    X_list = []
    meta_parts = []

    for Xspec, g in results:
        if Xspec is None or g is None:
            continue
        X_list.append(Xspec)
        meta_parts.append(g)

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    meta = pd.concat(meta_parts, axis=0).reset_index(drop=True)

    assert len(meta) == len(X), "Meta y spectral quedaron desalineados"

    gc.collect()
    return X, meta

In [12]:
def save_spectral_npz(X: np.ndarray, meta: pd.DataFrame, out_path: Path):
    np.savez_compressed(
        out_path,
        Xspec=X,
        path=np.array(meta["path"].astype(str).tolist()),
        orig_path=np.array(meta["orig_path"].astype(str).tolist()),
        clip_id=np.array(meta["clip_id"].astype(str).tolist()),
        speaker_id_final=np.array(meta["speaker_id_final"].astype(str).tolist()),
        clip_idx=meta["clip_idx"].to_numpy(np.int64),
        start_samp=meta["start_samp"].to_numpy(np.int64),
        end_samp=meta["end_samp"].to_numpy(np.int64),
        valence=meta["valence"].to_numpy(np.float32),
        arousal=meta["arousal"].to_numpy(np.float32),
        dominance=meta["dominance"].to_numpy(np.float32),
        win_rms=meta["win_rms"].to_numpy(np.float32),
        zrms_log=meta["zrms_log"].to_numpy(np.float32),
        lang=np.array(meta["lang"].astype(str).tolist()),
        is_aug=meta["is_aug"].to_numpy(bool),
        aug_ops=np.array(meta["aug_ops"].astype(str).tolist()),
        sr=np.int64(SR),
    )
    print("Guardado:", out_path, "| Xspec shape:", X.shape)

In [ ]:
VAL_WIN = WIN_DIR / "windows_val.npz"

Xspec_va, meta_spec_va = extract_spectral_features_fast(
    VAL_WIN,
    split_name="val",
    n_jobs=2
)


== Spectral features desde windows_val.npz ==


Spectral val:   0%|          | 0/2945 [00:00<?, ?it/s]

In [ ]:
save_spectral_npz(Xspec_va, meta_spec_va, OUT_DIR / "val_spectral.npz")

Guardado: /content/drive/MyDrive/pruebas/pre/spectral_feats/val_spectral.npz | Xspec shape: (8459, 4)


In [ ]:
TEST_WIN = WIN_DIR / "windows_test.npz"

Xspec_te, meta_spec_te = extract_spectral_features_fast(
    TEST_WIN,
    split_name="test",
    n_jobs=3
)


== Spectral features desde windows_test.npz ==


Spectral test:   0%|          | 0/3165 [00:00<?, ?it/s]

In [ ]:
save_spectral_npz(Xspec_te, meta_spec_te, OUT_DIR / "test_spectral.npz")

Guardado: /content/drive/MyDrive/pruebas/pre/spectral_feats/test_spectral.npz | Xspec shape: (9980, 4)


In [ ]:
TRAIN_WIN = WIN_DIR / "windows_train.npz"

Xspec_tr, meta_spec_tr = extract_spectral_features_fast(
    TRAIN_WIN,
    split_name="train",
    n_jobs=2
)


== Spectral features desde windows_train.npz ==


Spectral train:   0%|          | 0/11864 [00:00<?, ?it/s]

In [ ]:
save_spectral_npz(Xspec_tr, meta_spec_tr, OUT_DIR / "train_spectral.npz")

Guardado: /content/drive/MyDrive/pruebas/pre/spectral_feats/train_spectral.npz | Xspec shape: (38249, 4)


In [13]:
def compute_spectral_features_for_audio(y, sr, starts, ends, n_fft=400, hop_length=160):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))

    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.85)[0]
    flatness = librosa.feature.spectral_flatness(S=S)[0]

    feats = np.zeros((len(starts), 8), dtype=np.float32)

    for i, (s, e) in enumerate(zip(starts, ends)):
        f0 = int(s // hop_length)
        f1 = int(np.ceil(e / hop_length))
        f1 = max(f1, f0 + 1)

        c = centroid[f0:f1]
        b = bandwidth[f0:f1]
        r = rolloff[f0:f1]
        f = flatness[f0:f1]

        if len(c) > 0:
            feats[i, 0] = np.mean(c)
            feats[i, 1] = np.std(c)
            feats[i, 2] = np.mean(b)
            feats[i, 3] = np.std(b)
            feats[i, 4] = np.mean(r)
            feats[i, 5] = np.std(r)
            feats[i, 6] = np.mean(f)
            feats[i, 7] = np.std(f)

    return feats

In [ ]:
Xspec_va, meta_spec_va = extract_spectral_features_fast(
    VAL_WIN,
    split_name="val",
    n_jobs=2
)

save_spectral_npz(Xspec_va, meta_spec_va, OUT_DIR / "val_spectral.npz")


== Spectral features desde windows_val.npz ==


Spectral val:   0%|          | 0/2945 [00:00<?, ?it/s]

Guardado: /content/drive/MyDrive/pruebas/pre/spectral_feats/val_spectral.npz | Xspec shape: (8459, 8)


In [ ]:
Xspec_te, meta_spec_te = extract_spectral_features_fast(
    TEST_WIN,
    split_name="test",
    n_jobs=3
)

save_spectral_npz(Xspec_te, meta_spec_te, OUT_DIR / "test_spectral.npz")


== Spectral features desde windows_test.npz ==


Spectral test:   0%|          | 0/3165 [00:00<?, ?it/s]

Guardado: /content/drive/MyDrive/pruebas/pre/spectral_feats/test_spectral.npz | Xspec shape: (9980, 8)


In [ ]:
Xspec_tr, meta_spec_tr = extract_spectral_features_fast(
    TRAIN_WIN,
    split_name="train",
    n_jobs=3
)

save_spectral_npz(Xspec_tr, meta_spec_tr, OUT_DIR / "train_spectral.npz")


== Spectral features desde windows_train.npz ==


Spectral train:   0%|          | 0/11864 [00:00<?, ?it/s]

Guardado: /content/drive/MyDrive/pruebas/pre/spectral_feats/train_spectral.npz | Xspec shape: (38249, 8)


In [ ]:
D = np.load(OUT_DIR / "val_spectral.npz", allow_pickle=True)
print(D["Xspec"].shape)

In [14]:
def compute_spectral_features_for_audio(y, sr, starts, ends, n_fft=400, hop_length=160):
    """
    Devuelve 24 features por ventana:
      0-1   centroid mean/std
      2-3   bandwidth mean/std
      4-5   rolloff mean/std
      6-7   flatness mean/std

      8-9   delta centroid mean/std
      10-11 delta bandwidth mean/std
      12-13 delta rolloff mean/std
      14-15 delta flatness mean/std

      16-17 rms mean/std
      18-19 delta rms mean/std

      20-21 zcr mean/std
      22-23 delta zcr mean/std
    """
    import numpy as np
    import librosa

    # STFT-based features
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))

    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.85)[0]
    flatness = librosa.feature.spectral_flatness(S=S)[0]

    # Frame-level time-domain features aligned to hop_length
    rms = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop_length, center=True)[0]
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=n_fft, hop_length=hop_length, center=True)[0]

    # Deltas
    d_centroid = librosa.feature.delta(centroid[np.newaxis, :])[0]
    d_bandwidth = librosa.feature.delta(bandwidth[np.newaxis, :])[0]
    d_rolloff = librosa.feature.delta(rolloff[np.newaxis, :])[0]
    d_flatness = librosa.feature.delta(flatness[np.newaxis, :])[0]

    d_rms = librosa.feature.delta(rms[np.newaxis, :])[0]
    d_zcr = librosa.feature.delta(zcr[np.newaxis, :])[0]

    feats = np.zeros((len(starts), 24), dtype=np.float32)

    def _mstd(x):
        if len(x) == 0:
            return 0.0, 0.0
        return float(np.mean(x)), float(np.std(x))

    for i, (s, e) in enumerate(zip(starts, ends)):
        f0 = int(s // hop_length)
        f1 = int(np.ceil(e / hop_length))
        f1 = max(f1, f0 + 1)

        c = centroid[f0:f1]
        b = bandwidth[f0:f1]
        r = rolloff[f0:f1]
        f = flatness[f0:f1]

        dc = d_centroid[f0:f1]
        db = d_bandwidth[f0:f1]
        dr = d_rolloff[f0:f1]
        df = d_flatness[f0:f1]

        rm = rms[f0:f1]
        drm = d_rms[f0:f1]

        z = zcr[f0:f1]
        dz = d_zcr[f0:f1]

        feats[i, 0], feats[i, 1] = _mstd(c)
        feats[i, 2], feats[i, 3] = _mstd(b)
        feats[i, 4], feats[i, 5] = _mstd(r)
        feats[i, 6], feats[i, 7] = _mstd(f)

        feats[i, 8], feats[i, 9] = _mstd(dc)
        feats[i, 10], feats[i, 11] = _mstd(db)
        feats[i, 12], feats[i, 13] = _mstd(dr)
        feats[i, 14], feats[i, 15] = _mstd(df)

        feats[i, 16], feats[i, 17] = _mstd(rm)
        feats[i, 18], feats[i, 19] = _mstd(drm)

        feats[i, 20], feats[i, 21] = _mstd(z)
        feats[i, 22], feats[i, 23] = _mstd(dz)

    return feats

In [16]:
Xspec_va, meta_spec_va = extract_spectral_features_fast(
    VAL_WIN,
    split_name="val",
    n_jobs=3
)


== Spectral features desde windows_val.npz ==


Spectral val:   0%|          | 0/2945 [00:00<?, ?it/s]

In [17]:
save_spectral_npz(Xspec_va, meta_spec_va, OUT_DIR / "val_spectral3.npz")

Guardado: /content/drive/MyDrive/pruebas/pre/spectral_feats/val_spectral3.npz | Xspec shape: (8459, 24)


In [18]:
Xspec_te, meta_spec_te = extract_spectral_features_fast(
    TEST_WIN,
    split_name="test",
    n_jobs=3
)
save_spectral_npz(Xspec_te, meta_spec_te, OUT_DIR / "test_spectral.npz")


== Spectral features desde windows_test.npz ==


Spectral test:   0%|          | 0/3165 [00:00<?, ?it/s]

Guardado: /content/drive/MyDrive/pruebas/pre/spectral_feats/test_spectral.npz | Xspec shape: (9980, 24)


In [20]:
Xspec_tr, meta_spec_tr = extract_spectral_features_fast(
    TRAIN_WIN,
    split_name="train",
    n_jobs=4
)
save_spectral_npz(Xspec_tr, meta_spec_tr, OUT_DIR / "train_spectral3.npz")


== Spectral features desde windows_train.npz ==


Spectral train:   0%|          | 0/11864 [00:00<?, ?it/s]

Guardado: /content/drive/MyDrive/pruebas/pre/spectral_feats/train_spectral3.npz | Xspec shape: (38249, 24)
